# Map License — issue codes and build the mod

Everything here runs in your browser. You do not need Java, Python or Gradle
installed on your own computer.

Run the cells in order:

1. **Install Java** — one minute, once per session
2. **Load the mod source** — upload the folder you were given, as a zip
3. **Point the mod at your world** — which world the codes unlock
4. **Mint activation codes** — the codes you sell
5. **Build the jar** — the file you ship with your map
6. **Download** — jar and codes

Two kinds of code can be minted in step 4:

| | Pool code | Account-bound code |
| --- | --- | --- |
| Who can redeem it | any account | one account, and only that one |
| Mint in advance | yes | no — you need the buyer's username first |
| If it leaks | everyone who has it can use it | useless to anyone else |

Colab machines are wiped when you close them. **Download your codes before you
leave** — they cannot be recovered from the jar afterwards.

Full background is in `MAP-AUTHOR-GUIDE.txt` in the folder you were given.

---

*Licensing mod by Hi_Its_I.*


In [ ]:
#@title 1 · Install Java 21 { display-mode: "form" }
#@markdown Colab does not ship the Java version this mod needs, so it is
#@markdown fetched here. Takes about a minute. Run once per session.

import os
import subprocess

JDK_URL = ("https://api.adoptium.net/v3/binary/latest/21/ga/"
           "linux/x64/jdk/hotspot/normal/eclipse")

if not os.path.isdir("/opt/jdk21"):
    os.makedirs("/opt/jdk21", exist_ok=True)
    subprocess.run(f"curl -sL '{JDK_URL}' | tar -xz -C /opt/jdk21 --strip-components=1",
                   shell=True, check=True)

os.environ["JAVA_HOME"] = "/opt/jdk21"
os.environ["PATH"] = "/opt/jdk21/bin:" + os.environ["PATH"]

version = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(version.stderr.strip().splitlines()[0])
print("\nJava is ready. Go to step 2.")


In [ ]:
#@title 2 · Load the mod source { display-mode: "form" }
#@markdown Upload the `minecraft-map-license` folder as a **.zip** file, or
#@markdown clone it from a git repository if you keep it in one.

SOURCE = "Upload a zip file"  #@param ["Upload a zip file", "Clone a git repository"]
GIT_URL = ""  #@param {type:"string"}

import glob
import os
import shutil
import subprocess
import zipfile

WORKDIR = "/content/maplicense"
shutil.rmtree(WORKDIR, ignore_errors=True)
os.makedirs(WORKDIR, exist_ok=True)

if SOURCE == "Clone a git repository":
    if not GIT_URL.strip():
        raise SystemExit("Fill in GIT_URL, or switch SOURCE back to uploading a zip.")
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL.strip(), WORKDIR], check=True)
else:
    from google.colab import files

    print("Choose the zip of the mod folder you were given...")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("Nothing was uploaded.")

    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as archive:
        archive.extractall(WORKDIR)

# The zip may hold the folder itself or its contents; gradlew marks the root.
matches = glob.glob(f"{WORKDIR}/**/gradlew", recursive=True)
if not matches:
    raise SystemExit("No gradlew found in what you loaded - is this the mod folder?")

MOD_DIR = os.path.dirname(sorted(matches, key=len)[0])
os.chmod(f"{MOD_DIR}/gradlew", 0o755)

print(f"Mod source: {MOD_DIR}")
print("Go to step 3.")


In [ ]:
#@title 3 · Point the mod at your world { display-mode: "form" }
#@markdown Open your finished map with the mod installed and run
#@markdown `/maplicense fingerprint` as an operator. It prints the world's
#@markdown name and seed - copy them here.
#@markdown
#@markdown Any one of these matching is enough to lock the world, so filling in
#@markdown either is fine. Filling in both is better: a folder can be renamed,
#@markdown but the seed lives in level.dat.

MAP_ID = "my-story-map"  #@param {type:"string"}
MAP_TITLE = "My Story Map"  #@param {type:"string"}
WORLD_NAME = ""  #@param {type:"string"}
WORLD_SEED = ""  #@param {type:"string"}
#@markdown ---
#@markdown Any other text. Stops activation records from another build being
#@markdown accepted by yours. Change it once, before your first release.
INTEGRITY_KEY = "change-me-before-release"  #@param {type:"string"}

import json
import pathlib
import re

gate = {
    "mapId": MAP_ID.strip(),
    "title": MAP_TITLE.strip(),
    "markerFile": "map-license.json",
    "worldNames": [WORLD_NAME.strip().lower()] if WORLD_NAME.strip() else [],
    "seeds": [int(WORLD_SEED.strip())] if WORLD_SEED.strip() else [],
    "lockTimeoutSeconds": 300,
}

if not gate["worldNames"] and not gate["seeds"]:
    raise SystemExit("Fill in WORLD_NAME or WORLD_SEED - without one of them "
                     "the mod has no way to recognise your map.")

gate_path = pathlib.Path(MOD_DIR, "src/main/resources/map-license/gate.json")
gate_path.write_text(json.dumps(gate, indent="\t") + "\n", encoding="utf-8")

store = pathlib.Path(MOD_DIR, "src/main/java/net/jihoon/maplicense/LicenseStore.java")
text = store.read_text(encoding="utf-8")
patched, count = re.subn(r'(INTEGRITY_KEY\s*=\s*)"(?:[^"\\]|\\.)*"',
                         lambda m: m.group(1) + json.dumps(INTEGRITY_KEY), text, count=1)

if count:
    store.write_text(patched, encoding="utf-8")
    print(f'INTEGRITY_KEY set to "{INTEGRITY_KEY}"')
else:
    print("! Could not set INTEGRITY_KEY automatically - edit LicenseStore.java by hand.")

print(json.dumps(gate, indent=2))
print("\nSaved. Go to step 4.")


In [ ]:
#@title 4 · Mint activation codes { display-mode: "form" }
#@markdown Leave **BIND_TO_USERNAME** empty for codes any account can redeem.
#@markdown
#@markdown Put a buyer's Minecraft username in it to make a code that **only
#@markdown that account** can use - then a leaked code is useless to anyone
#@markdown else. Bound codes need one run per buyer.

HOW_MANY = 200  #@param {type:"integer"}
BIND_TO_USERNAME = ""  #@param {type:"string"}
BATCH_NAME = "batch1"  #@param {type:"string"}

import pathlib
import subprocess

codes_path = pathlib.Path(MOD_DIR, f"tools/codes-{BATCH_NAME}.txt")
command = [
    "python3", f"{MOD_DIR}/tools/generate_codes.py",
    "--count", str(HOW_MANY),
    "--out-codes", str(codes_path),
    "--out-hashes", f"{MOD_DIR}/src/main/resources/map-license/codes.json",
]

if BIND_TO_USERNAME.strip():
    command += ["--for-player", BIND_TO_USERNAME.strip()]

result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout or result.stderr)
result.check_returncode()

CODES_FILE = str(codes_path)
lines = [line for line in codes_path.read_text().splitlines()
         if line and not line.startswith("#")]

print(f"\nFirst few of {len(lines)}:")
for line in lines[:5]:
    print("   ", line)
if len(lines) > 5:
    print(f"    ... and {len(lines) - 5} more in the file you download in step 6")

print("\nGo to step 5 to build a jar that accepts them.")


In [ ]:
#@title 5 · Build the jar { display-mode: "form" }
#@markdown Takes a few minutes the first time - Gradle downloads Minecraft and
#@markdown the mappings. Later runs in the same session are much quicker.

import glob
import os
import subprocess

JAVA_HOME = os.environ.get("JAVA_HOME", "")
if not os.path.isdir(JAVA_HOME):
    JAVA_HOME = "/opt/jdk21"
if not os.path.isdir(JAVA_HOME):
    raise SystemExit("No Java found - run step 1 first, then come back here.")

env = dict(os.environ, JAVA_HOME=JAVA_HOME)
result = subprocess.run(["./gradlew", "clean", "build", "--no-daemon"],
                        cwd=MOD_DIR, env=env, capture_output=True, text=True)

if result.returncode != 0:
    print(result.stdout[-4000:])
    print(result.stderr[-4000:])
    raise SystemExit("Build failed - the output above says why.")

jars = [j for j in glob.glob(f"{MOD_DIR}/build/libs/*.jar") if "-sources" not in j]
JAR_FILE = sorted(jars)[-1]

print(f"Built {os.path.basename(JAR_FILE)}")
print("Go to step 6.")


In [ ]:
#@title 6 · Download the jar and your codes { display-mode: "form" }
#@markdown Three files come down: the jar to ship with your map, your codes,
#@markdown and the instruction sheet template to fill in for each buyer.
#@markdown
#@markdown **Save the codes somewhere permanent.** They exist nowhere else -
#@markdown they cannot be recovered from the jar, and this machine is wiped
#@markdown when you close it.

import os
import shutil
import zipfile

from google.colab import files

# Deliberately two files, named for what you do with them. Your codes must
# never end up inside the archive you hand to a buyer.
private_copy = f"/content/KEEP-PRIVATE-{os.path.basename(CODES_FILE)}"
shutil.copy(CODES_FILE, private_copy)

ship_zip = f"/content/to-ship-{os.path.basename(JAR_FILE).replace('.jar', '')}.zip"
with zipfile.ZipFile(ship_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(JAR_FILE, os.path.basename(JAR_FILE))
    sheet = os.path.join(MOD_DIR, "dist/READ-ME-FIRST.txt")
    if os.path.exists(sheet):
        archive.write(sheet, "READ-ME-FIRST.txt")

print(f"KEEP-PRIVATE-{os.path.basename(CODES_FILE)}  <- your codes, never send this")
print(f"{os.path.basename(ship_zip)}  <- jar + buyer instructions, safe to send
")
print("If the browser asks, allow multiple downloads.
")

files.download(private_copy)
files.download(ship_zip)

print("""
Next:

  1. Back up the codes file. There is no second copy.
  2. Fill in the four [PLACEHOLDERS] in READ-ME-FIRST.txt for each buyer,
     including that buyer's own code.
  3. Send the buyer a zip of: READ-ME-FIRST.txt, the jar, your world folder.

To mint more codes later, re-run steps 4 and 5 and ship the new jar - it keeps
accepting every code minted before it. Codes only work in a jar built after
they were minted.

One thing this notebook cannot do for you: wire your map's progression through
the `maplicense` scoreboard score, so that deleting the mod breaks the map
instead of unlocking it. MAP-AUTHOR-GUIDE.txt explains it under STEP 2.
""")
